# BELGOCAM SA - Forecast Q4 2026 (Volumes et Valeurs)

## Modélisation PACE - Notebook reproductible

**Auteur**: William Francis Fohom, Data Analyst, Administrateur National de Ventes
**Date**: 27 août 2026
**Période forecast**: Septembre - Décembre 2026

### Méthodologie PACE
1. **P**repare - Préparation des données
2. **A**nalyze - Analyse Exploratoire des Données (AED)
3. **C**onstruct - Construction du modèle Prophet
4. **E**xecute - Génération des livrables

### 4 Scénarios Soja
- **S1 - Rupture totale**: Ventes soja = 0 après le 16/09/2026
- **S2 - Réappro 50%**: 40 000 sacs au 01/10/2026
- **S3 - Réappro 100%**: 80 000 sacs au 15/09/2026 (référence)
- **S4 - Baisse prix**: S3 + baisse prix soja -10%

## 1. Phase PREPARE - Chargement des librairies et données

### 1.1 Librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import json
import os
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

# Font setup
fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Prophet
from prophet import Prophet

print('✓ Librairies chargées')

### 1.2 Référentiels produits et agences

Les références produits sont mappées à leur famille et leur poids unitaire (kg par unité de conditionnement).

In [ ]:
# Références produits
SOJA_REFS = {'T102': 50, 'T1021': 1, 'T1023': 5, 'T1024': 25}
CONC_REFS = {
    'C101': 50, 'C102': 50, 'C103': 50, 'C104': 50, 'C1042': 1, 'C1043': 5, 'C1044': 25,
    'C105': 50, 'C1053': 1, 'C1054': 5, 'C1055': 25, 'C108': 50,
}
MAIS_REFS = {'M1051': 50, 'M1052': 50}
INGREDIENT_REFS = {
    'B100': 25, 'E101': 25, 'I105': 25, 'B1001': 1, 'B1003': 5, 'B1004': 25,
    'E1011': 1, 'E1013': 5, 'E1014': 25, 'I1051': 1, 'I1053': 5, 'I1054': 25,
}
PREMIX_REFS = {'PX101': 25, 'PX102': 25, 'PX103': 25, 'PX104': 25, 'PX105': 25}
ALIMENT_REFS = {'CB100': 25, 'CB200': 25, 'PB100': 25, 'PB200': 25, 'DB100': 25, 'DB200': 25}

def get_family(ref):
    if ref in SOJA_REFS: return 'TOURTEAUX'
    if ref in CONC_REFS: return 'CONCENTRES'
    if ref in MAIS_REFS: return 'MAIS'
    if ref in INGREDIENT_REFS: return 'INGREDIENTS'
    if ref in PREMIX_REFS: return 'PREMIX'
    if ref in ALIMENT_REFS: return 'ALIMENT_COMPLET'
    return 'AUTRES'

def get_weight(ref):
    if ref in SOJA_REFS: return SOJA_REFS[ref]
    if ref in CONC_REFS: return CONC_REFS[ref]
    if ref in MAIS_REFS: return MAIS_REFS[ref]
    if ref in INGREDIENT_REFS: return INGREDIENT_REFS[ref]
    if ref in PREMIX_REFS: return PREMIX_REFS[ref]
    if ref in ALIMENT_REFS: return ALIMENT_REFS[ref]
    return 50

# Mapping agences (formats 2025 et 2026)
AGENCE_MAP = {
    'AGENCE FAMLA': ('Famla', 'Ouest'), 'Famla': ('Famla', 'Ouest'),
    'AGENCE MESSASSI': ('Messassi', 'Centre'), 'Messassi': ('Messassi', 'Centre'),
    'AGENCE BERTOUA': ('Bertoua', 'Centre'), 'Bertoua': ('Bertoua', 'Centre'),
    'AGENCE NDOBO': ('Ndobo', 'Littoral'), 'Ndobo': ('Ndobo', 'Littoral'),
    'AGENCE DJELENG': ('Djeleng', 'Ouest'), 'Djeleng': ('Djeleng', 'Ouest'),
    'AGENCE VILLAGE': ('Village', 'Littoral'), 'Village': ('Village', 'Littoral'),
    'AGENCE PK11': ('Pk11', 'Littoral'), 'Pk11': ('Pk11', 'Littoral'),
    'AGENCE NGAOUNDERE': ('Ngaoundere', 'Centre'), 'Ngaoundere': ('Ngaoundere', 'Centre'),
    'AGENCE AHALA': ('Ahala', 'Centre'), 'Ahala': ('Ahala', 'Centre'),
    'AGENCE NKONGSAMBA': ('Nkongsamba', 'Littoral'), 'Nkongsamba': ('Nkongsamba', 'Littoral'),
    'AGENCE NKOLBISSON': ('Nkolbisson', 'Centre'), 'Nkolbisson': ('Nkolbisson', 'Centre'),
    'AGENCE NKOABANG': ('Nkoabang', 'Centre'), 'Nkoabang': ('Nkoabang', 'Centre'),
    'AGENCE DE BAMENDA - DEPOT MBOUDA': ('Mbouda', 'Ouest'), 'Mbouda': ('Mbouda', 'Ouest'),
    'AGENCE BUEA': ('Buea', 'Littoral'), 'Buea': ('Buea', 'Littoral'),
}

INTERNAL_CLIENT_PATTERNS = ['SPC', 'PDC', 'COMPTOIR', 'EMANA']

print('✓ Référentiels chargés')

### 1.3 Sources de données

| Fichier | Période | Usage |
|---|---|---|
| 86d96135-...xlsx | Jan-Déc 2025 | Historique annuel |
| ventes janv a juin 2026.xlsx | Jan-Juin 2026 | S1 2026 |
| NJS GROUP ERP (9).xlsx | Juillet 2026 | Mois complet |
| NJS GROUP ERP (21).xlsx | 01-26/08/2026 | Août MTD |

In [ ]:
# Chargement du dataset consolidé
# (Le script pace_01_prepare_dataset.py construit ce fichier)
df = pd.read_csv('/home/z/my-project/scripts/dataset_consolide.csv', parse_dates=['date'])
print(f'Total records: {len(df):,}')
print(f'Période: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'\nPar source:')
print(df['source'].value_counts())
print(f'\nPar famille:')
print(df['family'].value_counts())

## 2. Phase ANALYZE - AED

### 2.1 Volumes mensuels par famille

In [ ]:
df['year_month'] = df['date'].dt.to_period('M')
monthly_family = df.groupby(['year_month', 'family'])['tonnes'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
monthly_family.plot(ax=ax, marker='o')
ax.set_title('BELGOCAM - Volumes mensuels par famille (2025 - août 2026)', fontweight='bold')
ax.set_xlabel('Mois')
ax.set_ylabel('Tonnes')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.show()

### 2.2 Saisonnalité mensuelle (base 2025)

Indice = 1.0 représente la moyenne annuelle. Un indice de 1.86 en octobre pour TOURTEAUX signifie que ce mois représente 86% de plus que la moyenne mensuelle.

In [ ]:
df_2025 = df[df['year'] == 2025].copy()
monthly_2025 = df_2025.groupby(['month', 'family'])['tonnes'].sum().unstack(fill_value=0)
annual_avg_2025 = df_2025.groupby('family')['tonnes'].sum() / 12
seasonal_index = monthly_2025.div(annual_avg_2025, axis=1).round(2)

fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
seasonal_index.plot(ax=ax, marker='o')
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
ax.set_title('Indice de saisonnalité mensuelle (base 2025)', fontweight='bold')
ax.set_xlabel('Mois')
ax.set_ylabel('Indice (1.0 = moyenne)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Juin', 'Juil', 'Août', 'Sep', 'Oct', 'Nov', 'Déc'])
ax.grid(True, alpha=0.3)
plt.show()

print('Part du Q4 (sept-déc) dans le volume annuel 2025:')
q4_share = df_2025[df_2025['month'].isin([9,10,11,12])].groupby('family')['tonnes'].sum() / df_2025.groupby('family')['tonnes'].sum() * 100
print(q4_share.round(1))

## 3. Phase CONSTRUCT - Modélisation Prophet

### 3.1 Configuration du modèle

**Stratégie**: 1 modèle Prophet par famille × région (13 modèles au total), puis désagrégation par produit × agence proportionnellement à la part historique.

**Avantages**:
- Réduit le nombre de modèles (130+ → 13)
- Meilleure robustesse statistique (plus de données par modèle)
- Préserve la structure produit × agence via les parts historiques

In [ ]:
def fit_prophet_fast(history_df, periods=4, freq='MS'):
    """Fit Prophet et forecast N periods."""
    if len(history_df) < 4:
        avg = history_df['y'].mean() if len(history_df) > 0 else 0
        future_dates = pd.date_range(start='2026-09-01', periods=periods, freq=freq)
        return pd.DataFrame({'ds': future_dates, 'yhat': [avg] * periods, 
                             'yhat_lower': [avg * 0.8] * periods, 'yhat_upper': [avg * 1.2] * periods})
    try:
        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.05,
            interval_width=0.8,
            mcmc_samples=0,
        )
        m.fit(history_df)
        future = m.make_future_dataframe(periods=periods, freq=freq, include_history=False)
        forecast = m.predict(future)
        return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    except Exception as e:
        avg_q4 = history_df[history_df['ds'].dt.month.isin([9,10,11,12])]['y'].mean()
        if pd.isna(avg_q4) or avg_q4 == 0:
            avg_q4 = history_df['y'].mean()
        future_dates = pd.date_range(start='2026-09-01', periods=periods, freq=freq)
        return pd.DataFrame({'ds': future_dates, 'yhat': [avg_q4] * periods,
                             'yhat_lower': [avg_q4 * 0.8] * periods, 'yhat_upper': [avg_q4 * 1.2] * periods})

# Exemple: Forecast TOURTEAUX × Ouest
monthly_fr = df.groupby(['family', 'region', 'year_month'])['tonnes'].sum().reset_index()
monthly_fr['date'] = monthly_fr['year_month'].dt.to_timestamp()

history = monthly_fr[(monthly_fr['family'] == 'TOURTEAUX') & (monthly_fr['region'] == 'Ouest')].copy()
history = history[['date', 'tonnes']].rename(columns={'date': 'ds', 'tonnes': 'y'}).sort_values('ds')
history['y'] = history['y'].clip(lower=0)

print(f'Historique TOURTEAUX × Ouest: {len(history)} mois')
print(history.tail())

forecast = fit_prophet_fast(history, periods=4, freq='MS')
print(f'\nForecast Q4 2026:')
print(forecast)

### 3.2 Application des 4 scénarios soja

| Scénario | Sept | Oct | Nov | Déc |
|---|---|---|---|---|
| S1 Rupture totale | 50% | 0% | 0% | 0% |
| S2 Réappro 50% | 50% | 70% | 70% | 100% |
| S3 Réappro 100% | 70% | 100% | 100% | 100% |
| S4 Baisse prix | S3 × 1.05 (Sept) puis × 1.10 |

In [ ]:
# Chargement du forecast complet (généré par pace_04_prophet_forecast.py)
fcst = pd.read_csv('/home/z/my-project/scripts/forecast_q4_2026.csv', parse_dates=['date'])

print(f'Forecast records: {len(fcst):,}')
print(f'Scénarios: {fcst["scenario"].unique()}')
print(f'\nSynthèse par scénario:')
synth = fcst.groupby('scenario').agg(
    tonnes=('tonnes', 'sum'),
    ca_m_fcfa=('ca_m_fcfa', 'sum'),
).round({'tonnes': 0, 'ca_m_fcfa': 1})
print(synth)

## 4. Phase EXECUTE - Visualisation des résultats

### 4.1 Comparaison des scénarios

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

# Volume par scénario
synth_t = fcst.groupby('scenario')['tonnes'].sum().round(0)
scenarios = ['S1_rupture', 'S2_reappro_50', 'S3_reappro_100', 'S4_baisse_prix']
labels = ['S1\nRupture', 'S2\nRéappro 50%', 'S3\nRéappro 100%', 'S4\nBaisse prix']
colors = ['#FCE4D6', '#FFF2CC', '#C6EFCE', '#BDD7EE']

axes[0].bar(labels, [synth_t[s] for s in scenarios], color=colors, edgecolor='black')
axes[0].set_title('Volume Q4 2026 par scénario (tonnes)', fontweight='bold')
axes[0].set_ylabel('Tonnes')
for i, s in enumerate(scenarios):
    axes[0].text(i, synth_t[s] + 500, f'{synth_t[s]:,.0f} t', ha='center', fontweight='bold')

# CA par scénario
synth_ca = fcst.groupby('scenario')['ca_m_fcfa'].sum().round(1)
axes[1].bar(labels, [synth_ca[s] for s in scenarios], color=colors, edgecolor='black')
axes[1].set_title('CA Q4 2026 par scénario (M FCFA)', fontweight='bold')
axes[1].set_ylabel('M FCFA')
for i, s in enumerate(scenarios):
    axes[1].text(i, synth_ca[s] + 200, f'{synth_ca[s]:,.0f} M', ha='center', fontweight='bold')

plt.show()

### 4.2 Évolution mensuelle par scénario

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

for scn, color, label in zip(scenarios, colors, labels):
    monthly_t = fcst[fcst['scenario'] == scn].groupby('month')['tonnes'].sum()
    ax.plot(monthly_t.index, monthly_t.values, marker='o', label=label, color=color, linewidth=2)

ax.set_title('Évolution mensuelle du volume Q4 2026 par scénario', fontweight='bold')
ax.set_xlabel('Mois')
ax.set_ylabel('Tonnes')
ax.set_xticks([9, 10, 11, 12])
ax.set_xticklabels(['Sept', 'Oct', 'Nov', 'Déc'])
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 5. Conclusion

Le forecast Q4 2026 présente une forte sensibilité au scénario de réapprovisionnement soja:

- **Scénario catastrophe (S1)**: 7 291 t, 4 656 M FCFA — impact direct de la rupture
- **Scénario réaliste (S3)**: 37 988 t, 17 304 M FCFA — référence pour la planification
- **Scénario optimiste (S4)**: 41 067 t, 17 168 M FCFA — la baisse de prix stimule le volume mais réduit le CA unitaire

### Recommandations
1. **Réapprovisionnement soja urgent** avant le 15/09/2026 (80 000 sacs min)
2. **Maintenir le plan d'action bundle** (ratio 2,4:1 atteint en août)
3. **Surveiller FAMLA et MESSASSI** qui sous-performent en août
4. **Préparer le plan commercial Q4** en s'appuyant sur le scénario S3

### Prochaines étapes
- Validation du forecast par la Direction Commerciale
- Suivi hebdomadaire de l'atteinte des projections
- Ajustement mensuel avec les nouvelles extractions ERP